# 03 · Claude Code eval: coding tasks inside Claude Code, on llama.cpp

The Q4 GGUF models solve coding tasks **inside Claude Code**, the way they are used at home:
- `llama-server` serves the GGUF. A small proxy (`eval/claude_code.py`) turns Claude Code's Anthropic API requests into
  OpenAI requests for llama-server and back.
- Claude Code runs headless (`claude -p`) with the **six tools used at home**: `Bash, Read, Edit, Write, Glob, Grep`
  (`--tools`). This keeps its system prompt and tool definitions at ~6k tokens instead of ~14k, so the training
  episodes of `04` fit the training length of `05`.
- Every run gets a fresh project folder. It is graded by the task's tests after the tests (and for JavaScript the
  package config) are restored, so editing the tests does not help.

**Task sets** (never used for training):

| Source | Tasks | Grader |
|---|---|---|
| `cc_kodcode_hard` | 100 hard KodCode tasks from the held-out split, disjoint from `agent_code` and round 1 | pytest |
| `polyglot_python` | 34 Exercism exercises of the [Aider polyglot benchmark](https://github.com/Aider-AI/polyglot-benchmark) | pytest |
| `polyglot_javascript` | 49 exercises of the same benchmark, all tests enabled | jest |

**When to run it:**
- **Now**, for the baseline: the base model (Unsloth's UD-Q4_K_XL) and RFT-Agent.
- **After `06`**, for the new model: add it to `MODELS` and `RUN_MODELS`. Finished models are skipped.

**Runtime and cost:**
- Runtime: **A100 80GB** (≈6.8 CU/hour).
- Setup ~20 min (llama.cpp build, Node.js, JavaScript packages), then ~1 hour per model with `SAMPLES = 1`.
  **~15–20 CU** for two models.
- Resumable: results are appended to Drive run by run. After a disconnect run the notebook again.

**Before you start:** copy the repo's current `eval/` folder to Drive (it has the new `claude_code.py`).

## Keep the browser tab alive (phone)

Colab drops the session when the browser freezes the tab, which happens when the phone screen turns off.
A tab that is playing audio is not frozen, so the cell below plays a very quiet tone in a loop. Run it first.
- If nothing plays, tap ▶ on the player. Chrome's media notification should appear.
- Turn the media volume down, but not to zero. Another app playing audio (e.g. Spotify) pauses it.
- Android (Xiaomi/HyperOS): set Chrome's battery saver to *No restrictions* and lock Chrome in recent apps.
- The player keeps playing after *Restart session*; if it stops, run the cell again.

Long steps still write to Drive and resume when rerun, so a dropped session only loses the step in progress.

In [ ]:
# Plays a very quiet 200 Hz tone in a loop so the browser does not freeze this tab
import numpy as np
from IPython.display import Audio, HTML, display

_sr = 8000
_wave = 0.01 * np.sin(2 * np.pi * 200 * np.arange(_sr * 5) / _sr)
display(HTML(Audio(_wave, rate=_sr, autoplay=True, normalize=False)._repr_html_().replace("<audio ", "<audio loop ")))

In [ ]:
# Node.js and Claude Code, pinned: every eval and the training data use the same Claude Code
import os
NODE_VERSION, CLAUDE_CODE_VERSION = "22.12.0", "2.1.331"
NODE_BIN = f"/opt/node-v{NODE_VERSION}-linux-x64/bin"
CLAUDE_BIN = f"{NODE_BIN}/claude"
!pip install -q pytest datasets
!curl -fsSL https://nodejs.org/dist/v{NODE_VERSION}/node-v{NODE_VERSION}-linux-x64.tar.xz | tar -xJ -C /opt
os.environ["PATH"] = f"{NODE_BIN}:{os.environ['PATH']}"
!npm install -g --silent @anthropic-ai/claude-code@{CLAUDE_CODE_VERSION}
!{CLAUDE_BIN} --version
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# llama.cpp server, CUDA build for this GPU only (~10 min)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build /content/llama.cpp/build -j --target llama-server 2>&1 | tail -3
LLAMA_SERVER = "/content/llama.cpp/build/bin/llama-server"

## Settings

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/colab-ai"
EVAL_DIR = f"{DRIVE_ROOT}/eval"
ROUND1_GEN = f"{DRIVE_ROOT}/rft/round1/generations.jsonl"
WORK = "/srv/cc_work"

# label -> (Hugging Face GGUF repo, text in the file name). The labels name the result files.
MODELS = {
    "base-q4": ("unsloth/Qwen3.6-35B-A3B-GGUF", "UD-Q4_K_XL"),
    "RFT-Agent-q4": ("salihyesil59/Qwen3.6-35B-A3B-RFT-Agent-GGUF", "Q4_K_M"),
    # after 06: "RFT-ClaudeCode-q4": ("salihyesil59/Qwen3.6-35B-A3B-RFT-ClaudeCode-GGUF", "Q4_K_M"),
}
RUN_MODELS = ["base-q4", "RFT-Agent-q4"]

KODCODE_HARD_N = 100
POLYGLOT_LANGS = ["python", "javascript"]
SAMPLES = 1                 # runs per task; 2 halves the noise and doubles the time
SLOTS = 24                  # parallel conversations on the server = runs at the same time
SLOT_CONTEXT = 32768        # tokens per conversation: ~6k Claude Code prompt + the episode
TURN_TOKENS = 16384         # max tokens of one reply
MAX_TURNS = 40              # Claude Code --max-turns
MINUTES = 20                # time limit of one run
SAMPLING = {"temperature": 0.6, "top_p": 0.95, "top_k": 20}   # the recommended thinking-mode settings
CU_PER_HOUR = 6.77
SEED = 3407

In [ ]:
import os, sys, json, glob, random, time, hashlib, importlib, collections, subprocess
from datetime import datetime
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
login(token=userdata.get("HF_TOKEN"))
for f in ("claude_code.py", "agent.py", "run_eval.py"):
    assert os.path.exists(f"{EVAL_DIR}/{f}"), f"{EVAL_DIR}/{f} not found: copy the repo's current eval/ folder to Drive"
sys.path.insert(0, EVAL_DIR)
sys.path_importer_cache.pop(EVAL_DIR, None)
importlib.invalidate_caches()
import run_eval as R
import agent as A
import claude_code as CC

# Claude Code and the model's commands run as the unprivileged user "agent", in /srv (outside /content).
# /content (with Google Drive under it) is closed to that user, so the model cannot touch Drive.
CC.ensure_user("agent")
os.makedirs(WORK, exist_ok=True)
os.chmod("/srv", 0o755)
os.chmod(WORK, 0o755)
os.chmod("/content", 0o700)
probe = subprocess.run(["runuser", "-u", "agent", "--", "ls", "/content/drive"], capture_output=True)
assert probe.returncode != 0, "the agent user can read /content/drive: do not run the model's commands"
print("Drive is closed to the agent user.")


def uid(prefix, text):
    return f"{prefix}-{hashlib.md5(text.encode()).hexdigest()[:12]}"


held_out = lambda qid: int(hashlib.md5(qid.encode()).hexdigest()[:2], 16) < 8   # the eval split of 02 (~3%)

## Task sets

- `cc_kodcode_hard`: built once and saved to Drive as `eval/tasks/cc_kodcode_hard.jsonl`, so every model gets the same tasks.
- Polyglot: the benchmark repo is cloned; the JavaScript packages (jest, babel) are installed once in `/srv/js_deps`
  and linked into every project, read-only for the agent user.

In [ ]:
from datasets import load_dataset

CC_EVAL_PATH = f"{EVAL_DIR}/tasks/cc_kodcode_hard.jsonl"
if os.path.exists(CC_EVAL_PATH):
    kodcode_hard = [json.loads(l) for l in open(CC_EVAL_PATH, encoding="utf-8")]
else:
    excluded = set()
    if os.path.exists(f"{EVAL_DIR}/tasks/agent_code.jsonl"):
        excluded |= {json.loads(l)["id"] for l in open(f"{EVAL_DIR}/tasks/agent_code.jsonl", encoding="utf-8")}
    if os.path.exists(ROUND1_GEN):
        excluded |= {json.loads(l)["id"] for l in open(ROUND1_GEN, encoding="utf-8")}
    kodcode_hard = []
    stream = load_dataset("KodCode/KodCode-V1", split="train", streaming=True).shuffle(seed=SEED + 3, buffer_size=20_000)
    for r in stream:
        qid = r["question_id"]
        if (r["style"] != "instruct" or r["gpt_difficulty"] != "hard" or not r["test_info"]
                or "from solution import" not in r["test"] or not held_out(qid)
                or uid("agent", qid) in excluded or uid("code", qid) in excluded):
            continue
        stub = A.make_stub(r["test_info"])
        if stub:
            kodcode_hard.append(CC.kodcode_task("cckh", r, stub, "cc_kodcode_hard"))
        if len(kodcode_hard) >= KODCODE_HARD_N:
            break
    with open(CC_EVAL_PATH, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(t, ensure_ascii=False) + "\n" for t in kodcode_hard)
print(f"cc_kodcode_hard: {len(kodcode_hard)} tasks")

In [ ]:
!rm -rf /content/polyglot-benchmark && git clone -q --depth 1 https://github.com/Aider-AI/polyglot-benchmark /content/polyglot-benchmark
!git -C /content/polyglot-benchmark log -1 --format="polyglot-benchmark %h %cs"
tasks = list(kodcode_hard)
if "python" in POLYGLOT_LANGS:
    tasks += CC.polyglot_tasks("/content/polyglot-benchmark", "python")
if "javascript" in POLYGLOT_LANGS:
    os.makedirs("/srv/js_deps", exist_ok=True)
    with open("/srv/js_deps/package.json", "w") as f:
        json.dump(CC.js_dev_dependencies("/content/polyglot-benchmark"), f, indent=1)
    !cd /srv/js_deps && npm install --no-audit --no-fund --loglevel=error
    !chmod -R a+rX,go-w /srv/js_deps
    tasks += CC.polyglot_tasks("/content/polyglot-benchmark", "javascript", node_modules="/srv/js_deps/node_modules")
print(collections.Counter(t["source"] for t in tasks))

## Runs

For every model in `RUN_MODELS`: download the GGUF, start llama-server, run all tasks, stop the server.
- The first time a model runs, a smoke test of two tasks comes first: it shows whether tool calls and thinking come
  through (see `/content/llama-server.log` if they do not).
- Results go to `eval/results/claude-code_<label>.jsonl` on Drive, one line per run, and a summary to `summary.csv`.
  A model whose runs are all there is skipped.

In [ ]:
if "proxy" in globals():   # rerun of this cell: the old proxy and server free their ports first
    proxy.close()
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait()
proxy = CC.Proxy(port=8100, sampling=SAMPLING, turn_tokens=TURN_TOKENS, seed=SEED)
results_dir = f"{EVAL_DIR}/results"
os.makedirs(results_dir, exist_ok=True)


def n_lines(path):
    return sum(1 for l in open(path, encoding="utf-8") if l.strip()) if os.path.exists(path) else 0


for label in RUN_MODELS:
    repo, pattern = MODELS[label]
    results = f"{results_dir}/claude-code_{label}.jsonl"
    if n_lines(results) >= len(tasks) * SAMPLES:
        print(f"{label}: all runs done, skipped")
        continue
    t0 = time.time()
    model_path = CC.download_gguf(repo, pattern)
    print(f"{label}: {model_path}")
    server = CC.start_llama_server(LLAMA_SERVER, model_path, SLOTS, SLOT_CONTEXT, seed=SEED)
    try:
        if not os.path.exists(results):
            smoke = f"/content/smoke_{label}.jsonl"
            if os.path.exists(smoke):
                os.remove(smoke)
            CC.run_tasks(tasks[:2], 1, proxy, smoke, CLAUDE_BIN, work_root=WORK, concurrency=2, max_turns=MAX_TURNS,
                         minutes=MINUTES, user="agent", label="smoke")
            runs = [json.loads(l) for l in open(smoke, encoding="utf-8")]
            for r in runs:
                print(f"  smoke {r['id']}: ok={r['ok']} turns={r['turns']} tool calls={r['tool_calls']} end={r['end'][:120]}")
            assert any(r["tool_calls"] for r in runs), "no tool calls came through: see /content/llama-server.log"
        CC.run_tasks(tasks, SAMPLES, proxy, results, CLAUDE_BIN, work_root=WORK, concurrency=SLOTS,
                     max_turns=MAX_TURNS, minutes=MINUTES, user="agent", label=label)
    finally:
        server.terminate()
        server.wait()
    if n_lines(results) >= len(tasks) * SAMPLES:
        R.write_summary(results_dir, datetime.now().strftime("%Y%m%d-%H%M%S"), f"claude-code_{label}",
                        f"{repo}:{pattern}", CC.summarize(results))
    hours = (time.time() - t0) / 3600
    print(f"{label}: {hours:.2f} h ≈ {hours * CU_PER_HOUR:.1f} CU")

## Comparison

Every model that has a results file on Drive, including those run in earlier sessions. `unfinished` = the run did
not end on its own (turn limit, time limit, reply cut at the token limit, or the conversation no longer fit).

In [ ]:
rows = {}
for path in sorted(glob.glob(f"{results_dir}/claude-code_*.jsonl")):
    rows[os.path.basename(path)[len("claude-code_"):-len(".jsonl")]] = CC.summarize(path)
sources = sorted({s for st in rows.values() for s in st})
print(f"{'model':<22} " + " ".join(f"{s:>20}" for s in sources) + f" {'all':>8}")
for label, st in rows.items():
    cells = [f"{st[s]['correct'] / st[s]['n']:>19.1%} " if s in st else f"{'-':>20}" for s in sources]
    total_n, total_ok = sum(x["n"] for x in st.values()), sum(x["correct"] for x in st.values())
    print(f"{label:<22} " + " ".join(cells) + f" {total_ok / total_n:>8.1%}")
print()
print(f"{'model':<22} {'turns':>6} {'tools':>6} {'tokens':>7} {'min':>5}  unfinished")
for label, st in rows.items():
    n = sum(x["n"] for x in st.values())
    ends = sum((x["ends"] for x in st.values()), collections.Counter())
    unfinished = n - ends.get("success", 0)
    print(f"{label:<22} {sum(x['turns'] for x in st.values()) / n:>6.1f} {sum(x['tool_calls'] for x in st.values()) / n:>6.1f} "
          f"{sum(x['tokens'] for x in st.values()) // n:>7} {sum(x['seconds'] for x in st.values()) / n / 60:>5.1f}  "
          f"{unfinished / n:.0%} {dict(ends.most_common(4))}")

## Finish: stop spending CU

In [ ]:
proxy.close()
from google.colab import runtime
runtime.unassign()